# exp98 — contacts-v1 train-set rollouts: interactive explorer

Explore the **1,000,000 rollouts** (1000 train targets × 1000 rollouts each) generated for [Open-Athena/MarinFold#98](https://github.com/Open-Athena/MarinFold/issues/98) with Eric's tuned contacts-v1 1.5B model (eval loss 2.7566).

Each rollout is a sampled contacts-v1 contact set; we score every rollout's precision/recall/F1 (all sep≥6) vs the document's ground-truth contacts, and save the best-recall & best-F1 rollouts verbatim.

**Run top-to-bottom.** The data lives in a GCS bucket, so the auth cell will prompt for a Google login with access to `marin-us-east5`.


In [ ]:
!pip -q install gcsfs pyarrow pandas matplotlib ipywidgets

In [ ]:
# Authenticate to GCS (the rollout data lives in gs://marin-us-east5).
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as e:
    print('not in Colab / already authed:', e)

import json, numpy as np, pandas as pd, gcsfs
import matplotlib.pyplot as plt
fs = gcsfs.GCSFileSystem()
BASE = "gs://marin-us-east5/protein-structure/MarinFold/exp98_rollouts_contacts_v1_train/runs/full"
SUMMARY = BASE + "/per_target_summary.parquet"

## Per-target summary (1000 targets)
One row per target: length `L`, `n_gt` ground-truth contacts, and the best-of-1000 `best_recall` / `best_f1` plus per-rollout means and timing.


In [ ]:
with fs.open(SUMMARY, 'rb') as f:
    t = pd.read_parquet(f)
print(len(t), 'targets')
t[['entry_id','L','n_gt','best_recall','best_f1','mean_rec','mean_gen_tokens','tokens_per_s']].describe().round(3)

## Overview — best-of-1000 accuracy


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col in zip(ax, ['best_recall','best_f1']):
    a.hist(t[col], bins=30, color='steelblue', edgecolor='white')
    a.axvline(t[col].mean(), color='crimson', ls='--', label=f'mean {t[col].mean():.2f}')
    a.set_xlabel(col + ' (best of 1000, all sep>=6)'); a.set_ylabel('# targets'); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(t.L, t.best_f1, s=8, alpha=.4, label='best F1')
ax[0].scatter(t.L, t.best_recall, s=8, alpha=.4, label='best recall')
ax[0].set_xlabel('L'); ax[0].set_ylabel('best of 1000'); ax[0].legend(); ax[0].set_title('accuracy vs length')
sc = ax[1].scatter(t.n_gt, t.best_f1, c=t.L, s=8, alpha=.5, cmap='viridis')
fig.colorbar(sc, ax=ax[1], label='L'); ax[1].set_xlabel('# GT contacts'); ax[1].set_ylabel('best F1'); ax[1].set_title('best F1 vs contact count')
plt.tight_layout(); plt.show()

## Drill into one target
Load a target's full 1000-rollout metrics and its saved best rollouts.
`explore(entry_id)` plots the per-rollout precision–recall cloud + F1 distribution and prints the best documents.


In [ ]:
def load_target(entry):
    with fs.open(f'{BASE}/rollout_metrics/{entry}.parquet', 'rb') as f:
        m = pd.read_parquet(f)
    with fs.open(f'{BASE}/best_rollouts/{entry}.json') as f:
        b = json.load(f)
    return m, b

def explore(entry):
    m, b = load_target(entry)
    print(f"{entry}  L={b['L']}  n_gt={b['n_gt']}  gt_by_band={b['gt_by_band']}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(m.all_rec, m.all_prec, s=6, alpha=.25)
    ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].set_title(f'{len(m)} rollouts (all sep>=6)')
    ax[1].hist(m.all_f1, bins=30, color='steelblue', edgecolor='white')
    ax[1].set_xlabel('F1'); ax[1].set_ylabel('# rollouts'); ax[1].set_title('F1 distribution')
    plt.tight_layout(); plt.show()
    print(f"best F1   : F1={b['best_f1']['f1']:.3f} prec={b['best_f1']['precision']:.3f} rec={b['best_f1']['recall']:.3f}")
    print(f"best recall: rec={b['best_recall']['recall']:.3f} prec={b['best_recall']['precision']:.3f} f1={b['best_recall']['f1']:.3f}")
    print('\n--- best-F1 rollout document (truncated) ---')
    print(b['best_f1']['document'][:600], '...')
    return m, b

# default: the highest best-F1 target
_top = t.sort_values('best_f1').iloc[-1].entry_id
_ = explore(_top)

### Interactive picker (Colab)


In [ ]:
try:
    import ipywidgets as W
    order = t.sort_values('best_f1', ascending=False)
    opts = [(f"{r.entry_id}  (L={int(r.L)}, bestF1={r.best_f1:.2f})", r.entry_id) for r in order.itertuples()]
    W.interact(lambda entry: explore(entry), entry=W.Dropdown(options=opts, description='target'))
except Exception as e:
    print('ipywidgets unavailable — call explore("AF-...") directly:', e)